In [1]:
csv_files = [
    '/Users/xinniemai/Desktop/mcmed_data/original_mcmed_csv/visits.csv',
    '/Users/xinniemai/Desktop/mcmed_data/original_mcmed_csv/meds.csv',
    '/Users/xinniemai/Desktop/mcmed_data/original_mcmed_csv/pmh.csv',
    '/Users/xinniemai/Desktop/mcmed_data/original_mcmed_csv/labs.csv'
]
for file in csv_files:
    with open(file, 'r') as f:
        line_count = sum(1 for _ in f)
    print(f"{file}: {line_count} lines")

/Users/xinniemai/Desktop/mcmed_data/original_mcmed_csv/visits.csv: 118386 lines
/Users/xinniemai/Desktop/mcmed_data/original_mcmed_csv/meds.csv: 235719 lines
/Users/xinniemai/Desktop/mcmed_data/original_mcmed_csv/pmh.csv: 755342 lines
/Users/xinniemai/Desktop/mcmed_data/original_mcmed_csv/labs.csv: 5706471 lines


In [2]:
import pandas as pd
import numpy as np
from helpers import *
from util import *
from disease_defination import DISEASE_DEFS

In [3]:
df_visits = pd.read_csv('/Users/xinniemai/Desktop/mcmed_data/original_mcmed_csv/visits.csv')
df_meds = pd.read_csv('/Users/xinniemai/Desktop/mcmed_data/original_mcmed_csv/meds.csv')
df_pmh = pd.read_csv('/Users/xinniemai/Desktop/mcmed_data/original_mcmed_csv/pmh.csv')
df_labs = pd.read_csv('/Users/xinniemai/Desktop/mcmed_data/original_mcmed_csv/labs.csv')

In [4]:
visits_history = df_visits.copy()
meds_history = df_meds.copy()
pmh_history = df_pmh.copy()

# Demographics

In [5]:
patient_visit_columns = ['subject_id', 'stay_id',
                         'age', 'gender', 'race', 'ethnicity',
                         'intime', 'outtime', 'in_year', 'edregtime', 'edouttime', 'admittime', 'dischtime', 'intime_icu', 'outtime_icu',
                         'ed_los', 'ed_los_hours', 'next_ed_visit_time', 'next_ed_visit_time_diff', 'next_ed_visit_time_diff_days',
                         'time_to_icu_transfer_hours',
                         'triage_temperature', 'triage_heartrate', 'triage_resprate', 'triage_o2sat', 'triage_sbp', 'triage_dbp', 'triage_acuity',
                         'triage_MAP']
df_patient_visits = pd.DataFrame(columns=patient_visit_columns)

# Set pandas display options to show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Display first 5 rows with all columns
display(df_patient_visits.head())

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP


In [6]:
basic_column_mapping = {
    'MRN': 'subject_id',
    'CSN': 'stay_id', 
    'Age': 'age',
    'Gender': 'gender',
    'Race': 'race',
    'Ethnicity': 'ethnicity'
}

df_patient_visits[list(basic_column_mapping.values())] = df_visits[list(basic_column_mapping.keys())]

triage_mapping = {
    'Triage_Temp': 'triage_temperature',
    'Triage_HR': 'triage_heartrate', 
    'Triage_RR': 'triage_resprate',
    'Triage_SpO2': 'triage_o2sat',
    'Triage_SBP': 'triage_sbp',
    'Triage_DBP': 'triage_dbp'
}

df_patient_visits[list(triage_mapping.values())] = df_visits[list(triage_mapping.keys())]


In [7]:
time_column_mapping = {
    'Arrival_time': 'intime',
    'Departure_time': 'outtime', 
    'Roomed_time': 'edregtime',
    'Dispo_time': 'edouttime'
}

for source_col, target_col in time_column_mapping.items():
    df_patient_visits[target_col] = adjust_time_by_subtracting_year(df_visits[source_col])

df_patient_visits['admittime'] = adjust_time_by_subtracting_year(df_visits[df_visits['ED_dispo'] == 'Inpatient']['Admit_time'])
df_patient_visits['intime_icu'] = adjust_time_by_subtracting_year(df_visits[df_visits['ED_dispo'] == 'ICU']['Admit_time'])

In [8]:
hosp_los_timedelta = pd.to_timedelta(df_visits['Hosp_LOS'], unit='D')
derived_time_operations = {
    'dischtime': lambda: df_patient_visits['admittime'] + hosp_los_timedelta,
    'outtime_icu': lambda: df_patient_visits['intime_icu'] + hosp_los_timedelta,
    'in_year': lambda: df_visits['Arrival_time'].str[:4].astype(int)
}

for col_name, operation in derived_time_operations.items():
    df_patient_visits[col_name] = operation()


In [9]:
visit_interval = pd.to_timedelta(df_visits['Hours_to_next_visit'], unit='h')
visit_metrics = {
    'ed_los': round(df_visits['ED_LOS'] / 24, 1),
    'ed_los_hours': df_visits['ED_LOS'],
    'next_ed_visit_time': df_patient_visits['intime'] + visit_interval,
    'next_ed_visit_time_diff': df_visits['Hours_to_next_visit'], 
    'next_ed_visit_time_diff_days': round(df_visits['Hours_to_next_visit'] / 24, 1)
}

for col_name, values in visit_metrics.items():
    df_patient_visits[col_name] = values

In [10]:
time_diff_to_icu = df_patient_visits['intime_icu'] - df_patient_visits['intime']
df_patient_visits['time_to_icu_transfer_hours'] = round(time_diff_to_icu.dt.total_seconds() / 3600, 2)

df_patient_visits['triage_acuity'] = df_visits['Triage_acuity'].str[0].replace('n', np.nan).astype('Int64')
df_patient_visits['triage_MAP'] = round((df_patient_visits['triage_sbp'] + 2 * df_patient_visits['triage_dbp']) / 3, 1)

In [11]:
display(df_patient_visits)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.0
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,36.8,98.0,14.0,98.0,103.0,66.0,2,78.3
2,99121566,99121037,63,F,Other,Non-Hispanic/Non-Latino,2040-03-11 03:06:51,2040-03-11 13:37:51,2240,2040-03-11 03:43:51,2040-03-11 08:31:51,NaT,NaT,2040-03-11 08:31:51,2040-03-27 08:31:51,0.4,10.52,NaT,NaN,NaN,5.42,37.2,96.0,18.0,98.0,91.0,61.0,3,71.0
3,99608739,99327434,90,F,White,Non-Hispanic/Non-Latino,2096-08-31 16:43:05,2096-08-31 21:06:05,2296,2096-08-31 17:56:05,2096-08-31 20:56:05,NaT,NaT,NaT,NaT,0.2,4.38,2097-06-07 00:43:05,6704.0,279.3,NaN,NaN,69.0,16.0,98.0,107.0,42.0,3,63.7
4,99608739,99502864,89,F,White,Non-Hispanic/Non-Latino,2097-06-07 05:08:05,2097-06-07 09:35:05,2297,2097-06-07 05:59:05,2097-06-07 08:55:05,NaT,NaT,NaT,NaT,0.2,4.45,NaT,NaN,NaN,NaN,36.4,65.0,17.0,98.0,103.0,57.0,3,72.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,41,F,Other,Non-Hispanic/Non-Latino,2022-09-23 02:03:34,2022-09-23 06:38:34,2222,2022-09-23 02:58:34,2022-09-23 05:46:34,NaT,NaT,NaT,NaT,0.2,4.58,2022-09-25 21:03:34,67.0,2.8,NaN,36.8,86.0,18.0,100.0,134.0,76.0,2,95.3
118381,99312340,99391376,37,M,Black or African American,Non-Hispanic/Non-Latino,2045-12-31 02:58:40,2045-12-31 06:30:40,2245,2045-12-31 03:52:40,2045-12-31 05:10:40,2045-12-31 05:10:40,2046-01-01 05:10:40,NaT,NaT,0.1,3.53,NaT,NaN,NaN,NaN,36.9,89.0,17.0,96.0,145.0,81.0,2,102.3
118382,99828357,99836129,22,M,Other,Hispanic/Latino,2034-09-07 02:27:43,2034-09-07 06:22:43,2234,2034-09-07 04:27:43,2034-09-07 05:04:43,NaT,NaT,NaT,NaT,0.2,3.92,NaT,NaN,NaN,NaN,36.8,85.0,14.0,99.0,144.0,97.0,2,112.7
118383,98964925,98979879,24,M,Other,Hispanic/Latino,2088-11-06 10:18:50,2088-11-06 14:19:50,2288,2088-11-06 11:22:50,2088-11-06 13:11:50,NaT,NaT,NaT,NaT,0.2,4.02,NaT,NaN,NaN,NaN,NaN,75.0,16.0,100.0,122.0,78.0,4,92.7


# Chief Complaint

In [12]:
complaint_dict = {
    "chiefcom_chest_pain": "chest pain|tight chest|pressure chest|angina|retrosternal pain",
    "chiefcom_abdominal_pain": "abdominal pain|abd pain|stomach ache|belly pain|tummy ache",
    "chiefcom_headache": "headache|lightheaded|migraine|cephalgia|pressure head",
    "chiefcom_shortness_of_breath": "breath",
    "chiefcom_back_pain": "back pain|lower back pain|lumbago|upper back pain",
    "chiefcom_cough": "cough|dry cough|productive cough|hacking cough",
    "chiefcom_nausea_vomiting": "nausea|vomit|vomiting|emesis|throwing up",
    "chiefcom_fever_chills": "fever|chill|temperature|feverish|shivering",
    "chiefcom_syncope": "syncope|faint|fainted|blackout|loss of consciousness",
    "chiefcom_dizziness": "dizz|dizzy|vertigo|lightheaded|spinning|off balance"
}

df_patient_visits['CC'] = df_visits['CC']
df_patient_visits = encode_chief_complaints(df_patient_visits, complaint_dict)
display(df_patient_visits)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.0,FALL,0,0,0,0,0,0,0,0,0,0
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,36.8,98.0,14.0,98.0,103.0,66.0,2,78.3,"ALTERED MENTAL STATUS,SHORTNESS OF BREATH",0,0,0,1,0,0,0,0,0,0
2,99121566,99121037,63,F,Other,Non-Hispanic/Non-Latino,2040-03-11 03:06:51,2040-03-11 13:37:51,2240,2040-03-11 03:43:51,2040-03-11 08:31:51,NaT,NaT,2040-03-11 08:31:51,2040-03-27 08:31:51,0.4,10.52,NaT,NaN,NaN,5.42,37.2,96.0,18.0,98.0,91.0,61.0,3,71.0,"ABNORMAL LAB,GENERAL WEAKNESS",0,0,0,0,0,0,0,0,0,0
3,99608739,99327434,90,F,White,Non-Hispanic/Non-Latino,2096-08-31 16:43:05,2096-08-31 21:06:05,2296,2096-08-31 17:56:05,2096-08-31 20:56:05,NaT,NaT,NaT,NaT,0.2,4.38,2097-06-07 00:43:05,6704.0,279.3,NaN,NaN,69.0,16.0,98.0,107.0,42.0,3,63.7,POSSIBLE DVT,0,0,0,0,0,0,0,0,0,0
4,99608739,99502864,89,F,White,Non-Hispanic/Non-Latino,2097-06-07 05:08:05,2097-06-07 09:35:05,2297,2097-06-07 05:59:05,2097-06-07 08:55:05,NaT,NaT,NaT,NaT,0.2,4.45,NaT,NaN,NaN,NaN,36.4,65.0,17.0,98.0,103.0,57.0,3,72.3,URINARY RETENTION,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,41,F,Other,Non-Hispanic/Non-Latino,2022-09-23 02:03:34,2022-09-23 06:38:34,2222,2022-09-23 02:58:34,2022-09-23 05:46:34,NaT,NaT,NaT,NaT,0.2,4.58,2022-09-25 21:03:34,67.0,2.8,NaN,36.8,86.0,18.0,100.0,134.0,76.0,2,95.3,"MULTIPLE COMPLAINTS,SUICIDAL",0,0,0,0,0,0,0,0,0,0
118381,99312340,99391376,37,M,Black or African American,Non-Hispanic/Non-Latino,2045-12-31 02:58:40,2045-12-31 06:30:40,2245,2045-12-31 03:52:40,2045-12-31 05:10:40,2045-12-31 05:10:40,2046-01-01 05:10:40,NaT,NaT,0.1,3.53,NaT,NaN,NaN,NaN,36.9,89.0,17.0,96.0,145.0,81.0,2,102.3,NaN,0,0,0,0,0,0,0,0,0,0
118382,99828357,99836129,22,M,Other,Hispanic/Latino,2034-09-07 02:27:43,2034-09-07 06:22:43,2234,2034-09-07 04:27:43,2034-09-07 05:04:43,NaT,NaT,NaT,NaT,0.2,3.92,NaT,NaN,NaN,NaN,36.8,85.0,14.0,99.0,144.0,97.0,2,112.7,ARM LACERATION,0,0,0,0,0,0,0,0,0,0
118383,98964925,98979879,24,M,Other,Hispanic/Latino,2088-11-06 10:18:50,2088-11-06 14:19:50,2288,2088-11-06 11:22:50,2088-11-06 13:11:50,NaT,NaT,NaT,NaT,0.2,4.02,NaT,NaN,NaN,NaN,NaN,75.0,16.0,100.0,122.0,78.0,4,92.7,ALCOHOL INTOXICATION,0,0,0,0,0,0,0,0,0,0


# Visit, Medication, Comorbidities Data

In [17]:
visits_history['Arrival_time'] = adjust_time_by_subtracting_year(visits_history['Arrival_time'])
meds_history[['Start_date','End_date']] = meds_history[['Start_date','End_date']].apply(adjust_time_by_subtracting_year)
pmh_history['Noted_date'] = adjust_time_by_subtracting_year(pmh_history['Noted_date'])

visits_history['Admit_service'] = visits_history['Admit_service'].astype(str).fillna('')

pmh_history['Code'] = pmh_history['Code'].astype(str)
pmh_history['Code'] = pmh_history['Code'].str.replace('.', '', regex=False)

comorbidity_codes = build_comorbidity_code_dictionary()

visits_history = visits_history.sort_values(by=['MRN', 'Arrival_time']).reset_index(drop=True)
meds_history.set_index('MRN', inplace=True)
pmh_history = pmh_history.sort_values(by=['MRN', 'Noted_date'])


In [248]:
visit_history_columns = [
    'n_ed_30d', 'n_ed_90d', 'n_ed_365d',
    'n_hosp_30d', 'n_hosp_90d', 'n_hosp_365d',
    'n_icu_30d', 'n_icu_90d', 'n_icu_365d'
]
visit_history_mapping = calculate_visit_history(visits_history)

df_patient_visits = pd.merge(
    df_patient_visits,
    visit_history_mapping,
    left_on=['subject_id', 'stay_id'],
    right_on=['MRN', 'CSN'],
    how='left'
).drop(columns=['MRN', 'CSN'])

In [249]:
med_history_columns = ['n_med', 'n_medrecon']
med_history_results = calculate_medication_history(visits_history, meds_history)

df_patient_visits = pd.merge(
    df_patient_visits,
    med_history_results,
    left_on=['subject_id', 'stay_id'],
    right_on=['MRN', 'CSN'],
    how='left'
).drop(columns=['MRN', 'CSN'])

In [250]:
comorbidity_results = calculate_comorbidities_batch(visits_history, pmh_history, comorbidity_codes)

df_patient_visits = pd.merge(
    df_patient_visits,
    comorbidity_results,
    left_on=['subject_id', 'stay_id'],
    right_on=['MRN', 'CSN'],
    how='left'
).drop(columns=['MRN', 'CSN'])

In [251]:
display(df_patient_visits)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness,n_ed_30d,n_ed_90d,n_ed_365d,n_hosp_30d,n_hosp_90d,n_hosp_365d,n_icu_30d,n_icu_90d,n_icu_365d,n_med,n_medrecon,cci_MI,cci_CHF,cci_PVD,cci_Stroke,cci_Dementia,cci_Pulmonary,cci_Rheumatic,cci_PUD,cci_Liver1,cci_DM1,cci_DM2,cci_Paralysis,cci_Renal,cci_Cancer1,cci_Liver2,cci_Cancer2,cci_HIV,eci_HIV,eci_Alcohol,eci_BloodLoss,eci_Arrhythmia,eci_Pulmonary,eci_Coagulopathy,eci_CHF,eci_Anemia,eci_Depression,eci_DM2,eci_DM1,eci_Drugs,eci_FluidsLytes,eci_HTN2,eci_HTN1,eci_Hypothyroid,eci_Liver,eci_Lymphoma,eci_Tumor2,eci_Obesity,eci_NeuroOther,eci_Paralysis,eci_PUD,eci_PVD,eci_Psychoses,eci_PHTN,eci_Renal,eci_Rheumatic,eci_Tumor1,eci_Valvular,eci_WeightLoss
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.0,FALL,False,False,False,False,False,False,False,False,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,36.8,98.0,14.0,98.0,103.0,66.0,2,78.3,"ALTERED MENTAL STATUS,SHORTNESS OF BREATH",False,False,False,True,False,False,False,False,False,False,0,0,0,0,0,0,0,0,0,6,12,0,1,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0,0,1,1,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0
2,99121566,99121037,63,F,Other,Non-Hispanic/Non-Latino,2040-03-11 03:06:51,2040-03-11 13:37:51,2240,2040-03-11 03:43:51,2040-03-11 08:31:51,NaT,NaT,2040-03-11 08:31:51,2040-03-27 08:31:51,0.4,10.52,NaT,NaN,NaN,5.42,37.2,96.0,18.0,98.0,91.0,61.0,3,71.0,"ABNORMAL LAB,GENERAL WEAKNESS",False,False,False,False,False,False,False,False,False,False,0,0,1,0,0,1,0,0,0,6,14,0,1,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0,0,1,1,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0
3,99608739,99327434,90,F,White,Non-Hispanic/Non-Latino,2096-08-31 16:43:05,2096-08-31 21:06:05,2296,2096-08-31 17:56:05,2096-08-31 20:56:05,NaT,NaT,NaT,NaT,0.2,4.38,2097-06-07 00:43:05,6704.0,279.3,NaN,NaN,69.0,16.0,98.0,107.0,42.0,3,63.7,POSSIBLE DVT,False,False,False,False,False,False,False,False,False,False,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,99608739,99502864,89,F,White,Non-Hispanic/Non-Latino,2097-06-07 05:08:05,2097-06-07 09:35:05,2297,2097-06-07 05:59:05,2097-06-07 08:55:05,NaT,NaT,NaT,NaT,0.2,4.45,NaT,NaN,NaN,NaN,36.4,65.0,17.0,98.0,103.0,57.0,3,72.3,URINARY RETENTION,False,False,False,False,False,False,False,False,False,False,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,41,F,Other,Non-Hispanic/Non-Latino,2022-09-23 02:03:34,2022-09-23 06:38:34,2222,2022-09-23 02:58:34,2022-09-23 

# Scores

In [252]:
add_triage_MAP(df_patient_visits)
add_score_REMS(df_patient_visits)
add_score_CART(df_patient_visits)
add_score_NEWS(df_patient_visits)
add_score_NEWS2(df_patient_visits)
add_score_MEWS(df_patient_visits)
add_score_CCI(df_patient_visits)

score_columns = ['stay_id', 'score_CCI', 'score_CART','score_REMS',  'score_NEWS', 'score_NEWS2', 'score_MEWS']
final_score_columns = [col for col in score_columns if col in df_patient_visits.columns]

Variable 'add_triage_MAP' successfully added
Variable 'Score_REMS' successfully added
Variable 'Score_CART' successfully added
Variable 'Score_NEWS' successfully added
Variable 'Score_NEWS2' successfully added
Variable 'Score_MEWS' successfully added
Variable 'add_score_CCI' successfully added


In [253]:
display(df_patient_visits)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness,n_ed_30d,n_ed_90d,n_ed_365d,n_hosp_30d,n_hosp_90d,n_hosp_365d,n_icu_30d,n_icu_90d,n_icu_365d,n_med,n_medrecon,cci_MI,cci_CHF,cci_PVD,cci_Stroke,cci_Dementia,cci_Pulmonary,cci_Rheumatic,cci_PUD,cci_Liver1,cci_DM1,cci_DM2,cci_Paralysis,cci_Renal,cci_Cancer1,cci_Liver2,cci_Cancer2,cci_HIV,eci_HIV,eci_Alcohol,eci_BloodLoss,eci_Arrhythmia,eci_Pulmonary,eci_Coagulopathy,eci_CHF,eci_Anemia,eci_Depression,eci_DM2,eci_DM1,eci_Drugs,eci_FluidsLytes,eci_HTN2,eci_HTN1,eci_Hypothyroid,eci_Liver,eci_Lymphoma,eci_Tumor2,eci_Obesity,eci_NeuroOther,eci_Paralysis,eci_PUD,eci_PVD,eci_Psychoses,eci_PHTN,eci_Renal,eci_Rheumatic,eci_Tumor1,eci_Valvular,eci_WeightLoss,score_REMS,score_CART,score_NEWS,score_NEWS2,score_MEWS,score_CCI
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.000000,FALL,False,False,False,False,False,False,False,False,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,9,0,0,1,5
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,36.8,98.0,14.0,98.0,103.0,66.0,2,78.333333,"ALTERED MENTAL STATUS,SHORTNESS OF BREATH",False,False,False,True,False,False,False,False,False,False,0,0,0,0,0,0,0,0,0,6,12,0,1,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0,0,1,1,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0,3,4,2,2,0,8
2,99121566,99121037,63,F,Other,Non-Hispanic/Non-Latino,2040-03-11 03:06:51,2040-03-11 13:37:51,2240,2040-03-11 03:43:51,2040-03-11 08:31:51,NaT,NaT,2040-03-11 08:31:51,2040-03-27 08:31:51,0.4,10.52,NaT,NaN,NaN,5.42,37.2,96.0,18.0,98.0,91.0,61.0,3,71.000000,"ABNORMAL LAB,GENERAL WEAKNESS",False,False,False,False,False,False,False,False,False,False,0,0,1,0,0,1,0,0,0,6,14,0,1,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0,0,1,1,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0,3,4,3,3,2,9
3,99608739,99327434,90,F,White,Non-Hispanic/Non-Latino,2096-08-31 16:43:05,2096-08-31 21:06:05,2296,2096-08-31 17:56:05,2096-08-31 20:56:05,NaT,NaT,NaT,NaT,0.2,4.38,2097-06-07 00:43:05,6704.0,279.3,NaN,NaN,69.0,16.0,98.0,107.0,42.0,3,63.666667,POSSIBLE DVT,False,False,False,False,False,False,False,False,False,False,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,10,13,1,1,1,5
4,99608739,99502864,89,F,White,Non-Hispanic/Non-Latino,2097-06-07 05:08:05,2097-06-07 09:35:05,2297,2097-06-07 05:59:05,2097-06-07 08:55:05,NaT,NaT,NaT,NaT,0.2,4.45,NaT,NaN,NaN,NaN,36.4,65.0,17.0,98.0,103.0,57.0,3,72.333333,URINARY RETENTION,False,False,False,False,False,False,False,False,False,False,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,8,9,1,1,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

# Lab Results

In [254]:
df_labs_processed = df_labs.copy()
test_column_name = 'Component_name'
csn_column_name = 'CSN'
value_column_name = 'Component_value'

In [255]:
def normalize_lab_test_names(df, test_col):
    """
    Normalize lab test names by combining similar tests into standardized groups.
    """
    df = df.copy()
    df['normalized_test'] = df[test_col]
    
    normalization_rules = {
        'PCO2, Arterial (Combined)': ['PCO2 (A)', 'POC:PCO2 (A), ISTAT', 'PCO2.ARTERIAL'],
        'PCO2, Venous (Combined)': ['PCO2.VENOUS', 'POC:PCO2 (V), ISTAT'],
        'pH, Arterial (Combined)': ['PH.ARTERIAL', 'POC:PH (A), ISTAT'],
        'pH, Venous (Combined)': ['PH.VENOUS', 'POC:PH (V), ISTAT'],
        'PO2, Arterial (Combined)': ['PO2 (A)', 'POC:PO2 (A), ISTAT', 'PO2.ARTERIAL'],
        'PO2, Venous (Combined)': ['PO2.VENOUS', 'POC:PO2 (V), ISTAT'],
        
        'Potassium (Combined)': ['POTASSIUM', 'POC:POTASSIUM, ISTAT', 'POTASSIUM.WHOLE BLD'],
        'Sodium (Combined)': ['SODIUM', 'POC:SODIUM, ISTAT', 'SODIUM.WHOLE BLD'],
        
        'Calcium (Combined)': r'CALCIUM',
        'eGFR (Combined)': r'EGFR',
        'LDL (Combined)': r'LDL',
        'Cholesterol, HDL (Combined)': r'CHOLESTEROL.*HDL',
        'Bilirubin, Unconjugated (Combined)': r'BILIRUBIN, UNCONJ',
        
        'HCT & Hemoglobin (Combined)': r'^HCT|HEMOGLOBIN',
        'ESR (Combined)': r'ESR',
        'NRBC (Combined)': r'NRBC',
        'Blasts (Combined)': r'BLASTS',
        
        'Basophil % (Combined)': r'BASOPHILS? \s* % \s* \( (AUTO|MANUAL) DIFF \)',
        'Basophil Absolute (Combined)': r'BASOPHILS?.*ABSOLUTE',
        'Eosinophil % (Combined)': r'EOSINOPHILS? \s* % \s* \( (AUTO|MANUAL) DIFF \)',
        'Eosinophil Absolute (Combined)': r'EOSINOPHILS?.*ABSOLUTE',
        'Monocyte % (Combined)': r'MONOCYTES? \s* % \s* \( (AUTO|MANUAL) DIFF \)',
        'Monocyte Absolute (Combined)': r'MONOCYTE.*ABSOLUTE',
        'Lymphocyte Absolute (Combined)': r'LYMPHOCYTES?.*ABSOLUTE',
        
        'Blood Typing & RH (Combined)': r'abo/rh|blood type verification',
        'Antibody Screen (Combined)': r'ANTIBODY SCREEN',
        'Hepatitis B Surface Antigen (Combined)': r'HBSAG|HEP B SURFACE',
        'Anti-HAV IgG (Combined)': r'ANTI-HAV IGG',
        'Anti-HBc (Combined)': r'ANTI-HBC',
        'Anti-HCV (Combined)': r'ANTI-HCV',
        
        'Adenovirus (Combined)': r'ADENOVIRUS',
        'Rhinovirus (Combined)': r'RHINOVIRUS',
        'Escherichia Coli (Combined)': r'ESCHERICHIA COLI',
        
        'Normal RBC Count (Combined)': r'NORMAL RBC ON .* COUNT',
        "RBC's in Small Square (Combined)": r"RBC'S IN SMALL SQUARE",
        'Basophil CSF (Combined)': r'BASOPHILS?.*CSF'
    }
    
    for normalized_name, patterns in normalization_rules.items():
        if isinstance(patterns, list):
            mask = df[test_col].isin(patterns)
        else:
            mask = df[test_col].str.contains(patterns, case=False, na=False, regex=True)
        
        df.loc[mask, 'normalized_test'] = normalized_name
    
    return df

In [256]:
def process_lab_data(df_labs, min_frequency=10000):
    """
    Process lab data: clean values, normalize test names, and filter by frequency.
    """
    df_processed = df_labs.copy()
    
    df_processed['Component_name'] = df_processed['Component_name'].str.strip()
    df_processed['Component_value'] = (df_processed['Component_value']
                                       .astype(str)
                                       .str.extract(r'(\d+\.?\d*)', expand=False))
    df_processed['Component_value'] = pd.to_numeric(df_processed['Component_value'], errors='coerce')
    
    df_processed = df_processed.dropna(subset=['Component_value'])
    
    df_processed = normalize_lab_test_names(df_processed, 'Component_name')
    
    test_counts = df_processed['normalized_test'].value_counts()
    high_frequency_tests = test_counts[test_counts > min_frequency].index.tolist()
    
    print(f"Found {len(high_frequency_tests)} tests with > {min_frequency} occurrences.")
    
    return df_processed[df_processed['normalized_test'].isin(high_frequency_tests)]

In [257]:
def merge_lab_data_with_visits(df_patient_visits, df_labs_wide, join_col='CSN'):
    """
    Merge lab data with patient visits, handling column conflicts.
    """
    lab_cols_to_add = df_labs_wide.columns.drop(join_col)
    
    conflicting_cols = [col for col in lab_cols_to_add if col in df_patient_visits.columns]
    if conflicting_cols:
        print(f"Warning: Dropping {len(conflicting_cols)} conflicting columns from master dataset.")
        df_patient_visits = df_patient_visits.drop(columns=conflicting_cols)
    
    df_merged = pd.merge(
        df_patient_visits, 
        df_labs_wide, 
        left_on='stay_id', 
        right_on=join_col, 
        how='left'
    )
    
    return df_merged.drop(columns=[join_col], errors='ignore')

In [258]:
df_labs_filtered = process_lab_data(df_labs, min_frequency=10000)

df_labs_aggregated = (df_labs_filtered
                      .groupby(['CSN', 'normalized_test'])['Component_value']
                      .mean()
                      .reset_index())

df_labs_wide = (df_labs_aggregated
                .pivot(index='CSN', columns='normalized_test', values='Component_value')
                .reset_index())

df_labs_wide.columns.name = None

df_patient_visits = merge_lab_data_with_visits(df_patient_visits, df_labs_wide)

print(f"Final dataset shape: {df_patient_visits.shape}")

/var/folders/gt/9x17m7412yd64tl8r0gwfpbh0000gn/T/ipykernel_11675/4251489468.py:58: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[test_col].str.contains(patterns, case=False, na=False, regex=True)
/var/folders/gt/9x17m7412yd64tl8r0gwfpbh0000gn/T/ipykernel_11675/4251489468.py:58: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[test_col].str.contains(patterns, case=False, na=False, regex=True)
/var/folders/gt/9x17m7412yd64tl8r0gwfpbh0000gn/T/ipykernel_11675/4251489468.py:58: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[test_col].str.contains(patterns, case=False, na=False, regex=True)


Found 68 tests with > 10000 occurrences.
Final dataset shape: (118385, 173)


In [259]:
display(df_patient_visits)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness,n_ed_30d,n_ed_90d,n_ed_365d,n_hosp_30d,n_hosp_90d,n_hosp_365d,n_icu_30d,n_icu_90d,n_icu_365d,n_med,n_medrecon,cci_MI,cci_CHF,cci_PVD,cci_Stroke,cci_Dementia,cci_Pulmonary,cci_Rheumatic,cci_PUD,cci_Liver1,cci_DM1,cci_DM2,cci_Paralysis,cci_Renal,cci_Cancer1,cci_Liver2,cci_Cancer2,cci_HIV,eci_HIV,eci_Alcohol,eci_BloodLoss,eci_Arrhythmia,eci_Pulmonary,eci_Coagulopathy,eci_CHF,eci_Anemia,eci_Depression,eci_DM2,eci_DM1,eci_Drugs,eci_FluidsLytes,eci_HTN2,eci_HTN1,eci_Hypothyroid,eci_Liver,eci_Lymphoma,eci_Tumor2,eci_Obesity,eci_NeuroOther,eci_Paralysis,eci_PUD,eci_PVD,eci_Psychoses,eci_PHTN,eci_Renal,eci_Rheumatic,eci_Tumor1,eci_Valvular,eci_WeightLoss,score_REMS,score_CART,score_NEWS,score_NEWS2,score_MEWS,score_CCI,ALBUMIN,ALKALINE PHOSPHATASE,ALT (SGPT),ANION GAP,AST (SGOT),BASOPHIL % (AUTO DIFF),"BILIRUBIN, TOTAL",BLOOD UREA NITROGEN (BUN),BLOOD: URINE (UA),"BNP, NT-PRO",Basophil Absolute (Combined),CHLORIDE,CO2,CREATININE,Calcium (Combined),EOSINOPHIL % (AUTO DIFF),Eosinophil Absolute (Combined),GLOBULIN,GLUCOSE,GLUCOSE: URINE (UA),HCT & Hemoglobin (Combined),HEMATOCRIT (HCT),HIGH SENSITIVITY TROPONIN,IMMATURE GRANULOCYTE % (AUTODIFF) WAM,"IMMATURE GRANULOCYTE, ABSOLUTE (AUTO DIFF) WAM",INFLUENZA A,INFLUENZA B,INR,KETONE: URINE (UA),LEUKOCYTE ESTERASE: URINE (UA),LIPASE,LYMPHOCYTE % (AUTO DIFF),Lymphocyte Absolute (Combined),MAGNESIUM,MEAN CORPUSCULAR VOLUME (MCV),MONOCYTE % (AUTO DIFF),Monocyte Absolute (Combined),NEUTROPHIL % (AUTO DIFF),"NEUTROPHIL, ABSOLUTE (AUTO DIFF)",NITRITE: URINE (UA),NRBC (Combined),PART. THROMBOPLASTIN TIME,"PCO2, Venous (Combined)",PH: URINE (UA),PLATELET COUNT (PLT),"PO2, Venous (Combined)","POC: BHCG, ISTAT QUANTITATIVE","POC: HCG, ISTAT QUALITATIVE","POC:BASE EXCESS, ISTAT",POC:GLUCOSE BY METER,"POC:HCO3 (V), ISTAT","POC:LACTATE, ISTAT","POC:O2 SATURATION, ISTAT VENOUS","POC:TCO2 (V), ISTAT","PROTEIN, TOTAL",PROTEIN: URINE (UA),PROTHROMBIN TIME,Potassium (Combined),RED BLOOD CELLS (RBC),RED CELL DISTRIBUTION WIDTH (RDW),RSV,SARS-COV-2 RNA,SPECIFIC GRAVITY: URINE (UA),Sodium (Combined),TROPONIN I,WHITE BLOOD CELLS (WBC),eGFR (Combined),"pH, Venous (Combined)"
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.000000,FALL,False,False,False,False,False,False,False,False,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,9,0,0,1,5,3.4,139.0,28.0,9.0,38.0,8.0,2.0,17.0,34.0,NaN,7.0,108.0,23.0,84.000000,9.250000,3.8,32.0,4.7,109.000000,34.0,17.500000,21.133333,NaN,5.0,4.0,NaN,NaN,NaN,34.0,NaN,NaN,13.1,1.11,NaN,70.033333,8.6,73.00,73.2,6.23,2.0,0.0,NaN,NaN,6.0,410.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.1,34.0,NaN,4.500000,3.016667,22.2,NaN,NaN,1.0200,140.0,17.0,8.1,NaN,NaN
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,36.8,98.0,14.0,98.0,103.0,66.0,2,78.333333,"ALTERED MENTAL STATUS,SHORTNESS OF BREATH",False,False,False,True,False,False,False,False,False,False,0,0,0,0,0,0,0,0,0,6,12,0,1,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0,0,1,1,0,0,1,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0,

# Outcome

In [260]:
visits_history_copy = df_visits.copy()

time_cols = ["Arrival_time", "Roomed_time", "Departure_time", "Dispo_time", "Admit_time"]

for col in time_cols:
    year = (
        visits_history_copy[col].str.extract(r"(\d{4})")[0]
        .dropna()
        .astype(int) - 200
    )
    
    decrypted_time = (
        year.astype(str) + 
        visits_history_copy[col].str[4:]
    )

    visits_history_copy[f"{col}_200"] = pd.to_datetime(
        decrypted_time,
        format="%Y-%m-%dT%H:%M:%SZ",  
        errors="coerce" 
    )

visits_history_copy

,MRN,CSN,Visit_no,Visits,Age,Gender,Race,Ethnicity,Means_of_arrival,Triage_Temp,Triage_HR,Triage_RR,Triage_SpO2,Triage_SBP,Triage_DBP,Triage_acuity,CC,ED_dispo,Hours_to_next_visit,Dispo_class_next_visit,ED_LOS,Hosp_LOS,DC_dispo,Payor_class,Admit_service,Dx_ICD9,Dx_ICD10,Dx_name,Arrival_time,Roomed_time,Dispo_time,Admit_time,Departure_time,Arrival_time_200,Roomed_time_200,Departure_time_200,Dispo_time_200,Admit_time_200
0,98880961,98874959,1,1,82,F,White,Non-Hispanic/Non-Latino,EMS,36.2,81.0,18.0,96.0,122.0,86.0,3-Urgent,FALL,Inpatient,NaN,NaN,6.13,1.0,Skilled Nursing Facility,Medicare,Medicine,873.20,S01.21XA,"Laceration of nose, initial encounter",2283-03-02T07:36:59Z,2283-03-02T07:36:59Z,2283-03-02T12:29:59Z,2283-03-02T12:29:59Z,2283-03-02T13:44:59Z,2083-03-02 07:36:59,2083-03-02 07:36:59,2083-03-02 13:44:59,2083-03-02 12:29:59,2083-03-02 12:29:59
1,99121566,99354408,1,2,59,F,Other,Non-Hispanic/Non-Latino,EMS,36.8,98.0,14.0,98.0,103.0,66.0,2-Emergent,"ALTERED MENTAL STATUS,SHORTNESS OF BREATH",Inpatient,4841.0,ICU,9.53,7.0,Skilled Nursing Facility,NaN,Emergency Medicine,780.97,R41.82,"Altered mental status, unspecified altered mental status type",2239-08-22T01:13:51Z,2239-08-22T01:16:51Z,2239-08-22T07:20:51Z,2239-08-22T07:20:51Z,2239-08-22T10:45:51Z,2039-08-22 01:13:51,2039-08-22 01:16:51,2039-08-22 10:45:51,2039-08-22 07:20:51,2039-08-22 07:20:51
2,99121566,99121037,2,2,63,F,Other,Non-Hispanic/Non-Latino,EMS,37.2,96.0,18.0,98.0,91.0,61.0,3-Urgent,"ABNORMAL LAB,GENERAL WEAKNESS",ICU,NaN,NaN,10.52,16.0,Skilled Nursing Facility,NaN,Critical Care,"038.9, 785.52, 995.92","A41.9, R65.21",Septic shock (CMS-HCC),2240-03-11T03:06:51Z,2240-03-11T03:43:51Z,2240-03-11T08:31:51Z,2240-03-11T08:31:51Z,2240-03-11T13:37:51Z,2040-03-11 03:06:51,2040-03-11 03:43:51,2040-03-11 13:37:51,2040-03-11 08:31:51,2040-03-11 08:31:51
3,99608739,99327434,1,2,90,F,White,Non-Hispanic/Non-Latino,Self,NaN,69.0,16.0,98.0,107.0,42.0,3-Urgent,POSSIBLE DVT,Discharge,6704.0,Discharge,4.38,NaN,Home/Work (includes foster care),Medicare,NaN,729.81,M79.89,Leg swelling,2296-08-31T16:43:05Z,2296-08-31T17:56:05Z,2296-08-31T20:56:05Z,NaN,2296-08-31T21:06:05Z,2096-08-31 16:43:05,2096-08-31 17:56:05,2096-08-31 21:06:05,2096-08-31 20:56:05,NaT
4,99608739,99502864,2,2,89,F,White,Non-Hispanic/Non-Latino,Self,36.4,65.0,17.0,98.0,103.0,57.0,3-Urgent,URINARY RETENTION,Discharge,NaN,Discharge,4.45,NaN,Home/Work (includes foster care),Medicare,NaN,788.20,R33.9,Urinary retention,2297-06-07T05:08:05Z,2297-06-07T05:59:05Z,2297-06-07T08:55:05Z,NaN,2297-06-07T09:35:05Z,2097-06-07 05:08:05,2097-06-07 05:59:05,2097-06-07 09:35:05,2097-06-07 08:55:05,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,3,5,41,F,Other,Non-Hispanic/Non-Latino,Self,36.8,86.0,18.0,100.0,134.0,76.0,2-Emergent,"MULTIPLE COMPLAINTS,SUICIDAL",Discharge,67.0,NaN,4.58,NaN,Home/Work (includes foster care),Medicare,NaN,V62.84,R45.851,Suicidal ideation,2222-09-23T02:03:34Z,2222-09-23T02:58:34Z,2222-09-23T05:46:34Z,NaN,2222-09-23T06:38:34Z,2022-09-23 02:03:34,2022-09-23 02:58:34,2022-09-23 06:38:34,2022-09-23 05:46:34,NaT
118381,99312340,99391376,1,1,37,M,Black or African American,Non-Hispanic/Non-Latino,EMS,36.9,89.0,17.0,96.0,145.0,81.0,2-Emergent,NaN,Inpatient,NaN,NaN,3.53,1.0,Home/Work (includes foster care),Medicare,General Surgery,959.09,S09.93XA,"Facial injury, initial encounter",2245-12-31T02:58:40Z,2245-12-31T03:52:40Z,2245-12-31T05:10:40Z,2245-12-31T05:10:40Z,2245-12-31T06:30:40Z,2045-12-31 02:58:40,2045-12-31 03:52:40,2045-12-31 06:30:40,2045-12-31 05:10:40,2045-12-31 05:10:40
118382,99828357,99836129,1,1,22,M,Other,Hispanic/Latino,EMS,36.8,85.0,14.0,99.0,144.0,97.0,2-Emergent,ARM LACERATION,Discharge,NaN,NaN,3.92,NaN,Home/Work (includes foster care),NaN,NaN,884.0,S41.111A,"Laceration of arm, right, initial encounter",2234-09-07T02:27:43Z,2234-09-07T04:27:43Z,2234-09-07T05:04:4

In [261]:
pmh_history_copy = df_pmh.copy()

time_cols = ["Noted_date"]

for col in time_cols:
    year = (
        pmh_history_copy[col].str.extract(r"(\d{4})")[0]
        .dropna()
        .astype(int) - 200
    )
    
    decrypted_time = (
        year.astype(str) + 
        pmh_history_copy[col].str[4:]
    )
    
    pmh_history_copy[f"{col}_200"] = pd.to_datetime(
        decrypted_time,
        format="%Y-%m-%dT%H:%M:%SZ", 
        errors="coerce" 
    )

pmh_history_copy

,MRN,Noted_date,CodeType,Code,Desc10,CCS,DescCCS,Noted_date_200
0,98816154,2288-11-16T00:00:00Z,Dx10,M954,Acquired deformity of chest and rib,209.0,Other acquired deformities,2088-11-16
1,98816154,2293-06-16T00:00:00Z,Dx10,F411,Generalized anxiety disorder,651.0,Anxiety disorders,2093-06-16
2,98816154,2293-06-16T00:00:00Z,Dx10,F329,"Major depressive disorder, single episode, unspecified",657.0,Mood disorders,2093-06-16
3,98816163,2252-04-19T00:00:00Z,Dx10,R7989,Other specified abnormal findings of blood chemistry,258.0,Other screening for suspected conditions (not mental disorders or infectious disease),2052-04-19
4,98816163,2252-04-30T00:00:00Z,Dx10,C8590,"Non-Hodgkin lymphoma, unspecified, unspecified site",38.0,Non-Hodgkin`s lymphoma,2052-04-30
...,...,...,...,...,...,...,...,...
755336,99999964,2229-04-27T00:00:00Z,Dx10,R519,NaN,NaN,NaN,2029-04-27
755337,99999964,2229-06-22T00:00:00Z,Dx10,K631,Perforation of intestine (nontraumatic),155.0,Other gastrointestinal disorders,2029-06-22
755338,99999976,2233-11-25T00:00:00Z,Dx10,D509,"Iron deficiency anemia, unspecified",59.0,Deficiency and other anemia,2033-11-25
755339,99999976,2233-11-25T00:00:00Z,Dx10,R591,Generalized enlarged lymph nodes,247.0,Lymphadenitis,2033-11-25


In [262]:
pmh_history_copy["Noted_date_200"] = pd.to_datetime(pmh_history_copy["Noted_date_200"])
print(pmh_history_copy.dtypes) 

MRN                        int64
Noted_date                object
CodeType                  object
Code                      object
Desc10                    object
CCS                      float64
DescCCS                   object
Noted_date_200    datetime64[ns]
dtype: object


In [263]:
pmh_history_copy['icd_version'] = pmh_history_copy['CodeType'].apply(lambda x: 9 if 'Dx9' in x else 10)

pmh_history_copy['ICD_10_Code'] = pmh_history_copy.apply(
    lambda row: convert_9to10(row['Code']) if row['icd_version'] == 9 else row['Code'],
    axis=1
)
pmh_history_copy['icd_version'] = 10

pmh_history_copy["ICD_9_Code"] = pmh_history_copy.apply(
    lambda row: convert_10to9(row["ICD_10_Code"]), axis=1
)

pmh_history_copy.head()


,MRN,Noted_date,CodeType,Code,Desc10,CCS,DescCCS,Noted_date_200,icd_version,ICD_10_Code,ICD_9_Code
0,98816154,2288-11-16T00:00:00Z,Dx10,M954,Acquired deformity of chest and rib,209.0,Other acquired deformities,2088-11-16,10,M954,7383
1,98816154,2293-06-16T00:00:00Z,Dx10,F411,Generalized anxiety disorder,651.0,Anxiety disorders,2093-06-16,10,F411,30002
2,98816154,2293-06-16T00:00:00Z,Dx10,F329,"Major depressive disorder, single episode, unspecified",657.0,Mood disorders,2093-06-16,10,F329,311
3,98816163,2252-04-19T00:00:00Z,Dx10,R7989,Other specified abnormal findings of blood chemistry,258.0,Other screening for suspected conditions (not mental disorders or infectious disease),2052-04-19,10,R7989,79099
4,98816163,2252-04-30T00:00:00Z,Dx10,C8590,"Non-Hodgkin lymphoma, unspecified, unspecified site",38.0,Non-Hodgkin`s lymphoma,2052-04-30,10,C8590,20280


In [264]:

visits_history_copy['Dx_ICD9_list']  = visits_history_copy['Dx_ICD9'].apply(split_clean_icd)
visits_history_copy['Dx_ICD10_list'] = visits_history_copy['Dx_ICD10'].apply(split_clean_icd)

visits_history_copy['Dx_name_clean'] = visits_history_copy['Dx_name'].fillna('')

visits_history_copy['Dx_name_clean'] = visits_history_copy['Dx_name_clean'].str.lower()

visits_history_copy['Dx_name_clean'] = visits_history_copy['Dx_name_clean'].str.replace(r'\(.*?\)', '', regex=True).str.strip()
visits_history_copy

,MRN,CSN,Visit_no,Visits,Age,Gender,Race,Ethnicity,Means_of_arrival,Triage_Temp,Triage_HR,Triage_RR,Triage_SpO2,Triage_SBP,Triage_DBP,Triage_acuity,CC,ED_dispo,Hours_to_next_visit,Dispo_class_next_visit,ED_LOS,Hosp_LOS,DC_dispo,Payor_class,Admit_service,Dx_ICD9,Dx_ICD10,Dx_name,Arrival_time,Roomed_time,Dispo_time,Admit_time,Departure_time,Arrival_time_200,Roomed_time_200,Departure_time_200,Dispo_time_200,Admit_time_200,Dx_ICD9_list,Dx_ICD10_list,Dx_name_clean
0,98880961,98874959,1,1,82,F,White,Non-Hispanic/Non-Latino,EMS,36.2,81.0,18.0,96.0,122.0,86.0,3-Urgent,FALL,Inpatient,NaN,NaN,6.13,1.0,Skilled Nursing Facility,Medicare,Medicine,873.20,S01.21XA,"Laceration of nose, initial encounter",2283-03-02T07:36:59Z,2283-03-02T07:36:59Z,2283-03-02T12:29:59Z,2283-03-02T12:29:59Z,2283-03-02T13:44:59Z,2083-03-02 07:36:59,2083-03-02 07:36:59,2083-03-02 13:44:59,2083-03-02 12:29:59,2083-03-02 12:29:59,[87320],[S0121XA],"laceration of nose, initial encounter"
1,99121566,99354408,1,2,59,F,Other,Non-Hispanic/Non-Latino,EMS,36.8,98.0,14.0,98.0,103.0,66.0,2-Emergent,"ALTERED MENTAL STATUS,SHORTNESS OF BREATH",Inpatient,4841.0,ICU,9.53,7.0,Skilled Nursing Facility,NaN,Emergency Medicine,780.97,R41.82,"Altered mental status, unspecified altered mental status type",2239-08-22T01:13:51Z,2239-08-22T01:16:51Z,2239-08-22T07:20:51Z,2239-08-22T07:20:51Z,2239-08-22T10:45:51Z,2039-08-22 01:13:51,2039-08-22 01:16:51,2039-08-22 10:45:51,2039-08-22 07:20:51,2039-08-22 07:20:51,[78097],[R4182],"altered mental status, unspecified altered mental status type"
2,99121566,99121037,2,2,63,F,Other,Non-Hispanic/Non-Latino,EMS,37.2,96.0,18.0,98.0,91.0,61.0,3-Urgent,"ABNORMAL LAB,GENERAL WEAKNESS",ICU,NaN,NaN,10.52,16.0,Skilled Nursing Facility,NaN,Critical Care,"038.9, 785.52, 995.92","A41.9, R65.21",Septic shock (CMS-HCC),2240-03-11T03:06:51Z,2240-03-11T03:43:51Z,2240-03-11T08:31:51Z,2240-03-11T08:31:51Z,2240-03-11T13:37:51Z,2040-03-11 03:06:51,2040-03-11 03:43:51,2040-03-11 13:37:51,2040-03-11 08:31:51,2040-03-11 08:31:51,"[0389, 78552, 99592]","[A419, R6521]",septic shock
3,99608739,99327434,1,2,90,F,White,Non-Hispanic/Non-Latino,Self,NaN,69.0,16.0,98.0,107.0,42.0,3-Urgent,POSSIBLE DVT,Discharge,6704.0,Discharge,4.38,NaN,Home/Work (includes foster care),Medicare,NaN,729.81,M79.89,Leg swelling,2296-08-31T16:43:05Z,2296-08-31T17:56:05Z,2296-08-31T20:56:05Z,NaN,2296-08-31T21:06:05Z,2096-08-31 16:43:05,2096-08-31 17:56:05,2096-08-31 21:06:05,2096-08-31 20:56:05,NaT,[72981],[M7989],leg swelling
4,99608739,99502864,2,2,89,F,White,Non-Hispanic/Non-Latino,Self,36.4,65.0,17.0,98.0,103.0,57.0,3-Urgent,URINARY RETENTION,Discharge,NaN,Discharge,4.45,NaN,Home/Work (includes foster care),Medicare,NaN,788.20,R33.9,Urinary retention,2297-06-07T05:08:05Z,2297-06-07T05:59:05Z,2297-06-07T08:55:05Z,NaN,2297-06-07T09:35:05Z,2097-06-07 05:08:05,2097-06-07 05:59:05,2097-06-07 09:35:05,2097-06-07 08:55:05,NaT,[78820],[R339],urinary retention
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,3,5,41,F,Other,Non-Hispanic/Non-Latino,Self,36.8,86.0,18.0,100.0,134.0,76.0,2-Emergent,"MULTIPLE COMPLAINTS,SUICIDAL",Discharge,67.0,NaN,4.58,NaN,Home/Work (includes foster care),Medicare,NaN,V62.84,R45.851,Suicidal ideation,2222-09-23T02:03:34Z,2222-09-23T02:58:34Z,2222-09-23T05:46:34Z,NaN,2222-09-23T06:38:34Z,2022-09-23 02:03:34,2022-09-23 02:58:34,2022-09-23 06:38:34,2022-09-23 05:46:34,NaT,[V6284],[R45851],suicidal ideation
118381,99312340,99391376,1,1,37,M,Black or African American,Non-Hispanic/Non-Latino,EMS,36.9,89.0,17.0,96.0,145.0,81.0,2-Emergent,NaN,Inpatient,NaN,NaN,3.53,1.0,Home/Work (includes foster care),Medicare,General Surgery,959.09,S09.93XA,"Facial injury, initial encounter",2245-12-31T02:58:40Z,2245-12-31T03:52:40Z,2245-12-31T05:10:40Z,2245-12-31T05:10:40Z,2245-12-31T06:30:40Z,2045-12-31 02:58:40,2045-12-31 03:52:40,2045-12-31 06:30:40,2045-1

In [265]:
for disease_name, disease_def in DISEASE_DEFS.items():
    codes = [c.upper() for c in disease_def['codes']]
    keywords = disease_def['keywords']

    match_icd9 = visits_history_copy['Dx_ICD9_list'].apply(lambda lst: any(any(code and c.startswith(code) for code in codes) for c in lst))
    match_icd10 = visits_history_copy['Dx_ICD10_list'].apply(lambda lst: any(any(code and c.startswith(code) for code in codes) for c in lst))

    if keywords:
        keyword_pattern = '|'.join(keywords)
        match_kw = visits_history_copy['Dx_name_clean'].str.contains(keyword_pattern, case=False, na=False, regex=True)
    else:
        match_kw = False

    match_any = match_icd9 | match_icd10 | match_kw

    hit_ids = set(visits_history_copy.loc[match_any, 'CSN'])

    visits_history_copy[f'outcome_{disease_name}'] = visits_history_copy['CSN'].isin(hit_ids)

    print(f"{disease_name}: Matched {len(hit_ids)} CSNs, Total positive cases: {visits_history_copy[f'outcome_{disease_name}'].sum()}")

sepsis: Matched 843 CSNs, Total positive cases: 843
copd_exac: Matched 221 CSNs, Total positive cases: 221
acs_mi: Matched 926 CSNs, Total positive cases: 926
stroke: Matched 1356 CSNs, Total positive cases: 1356
ards: Matched 37 CSNs, Total positive cases: 37
aki: Matched 739 CSNs, Total positive cases: 739
pe: Matched 473 CSNs, Total positive cases: 473
pneumonia_bacterial: Matched 2 CSNs, Total positive cases: 2
pneumonia_viral: Matched 293 CSNs, Total positive cases: 293
pneumonia_all: Matched 295 CSNs, Total positive cases: 295
asthma_acute_exac: Matched 293 CSNs, Total positive cases: 293
ahf: Matched 991 CSNs, Total positive cases: 991


In [266]:
outcome_cols = [col for col in visits_history_copy.columns if col.startswith('outcome_')]
proportions = visits_history_copy[outcome_cols].mean().round(9)

print("Outcome column proportions (proportion with value = 1):")
print(proportions)

Outcome column proportions (proportion with value = 1):
outcome_sepsis                 0.007121
outcome_copd_exac              0.001867
outcome_acs_mi                 0.007822
outcome_stroke                 0.011454
outcome_ards                   0.000313
outcome_aki                    0.006242
outcome_pe                     0.003995
outcome_pneumonia_bacterial    0.000017
outcome_pneumonia_viral        0.002475
outcome_pneumonia_all          0.002492
outcome_asthma_acute_exac      0.002475
outcome_ahf                    0.008371
dtype: float64


In [267]:
visits_history_copy["outcome_hospitalization"] = (
    visits_history_copy["Admit_time"].notna() & 
    (visits_history_copy["ED_dispo"] == "Inpatient")
).astype(int)
visits_history_copy["outcome_critical"] = (visits_history_copy["ED_dispo"] == "ICU").astype(int)

visits_history_copy["outcome_discharge"] = (visits_history_copy["ED_dispo"] == "Discharge").astype(int)

visits_history_copy

,MRN,CSN,Visit_no,Visits,Age,Gender,Race,Ethnicity,Means_of_arrival,Triage_Temp,Triage_HR,Triage_RR,Triage_SpO2,Triage_SBP,Triage_DBP,Triage_acuity,CC,ED_dispo,Hours_to_next_visit,Dispo_class_next_visit,ED_LOS,Hosp_LOS,DC_dispo,Payor_class,Admit_service,Dx_ICD9,Dx_ICD10,Dx_name,Arrival_time,Roomed_time,Dispo_time,Admit_time,Departure_time,Arrival_time_200,Roomed_time_200,Departure_time_200,Dispo_time_200,Admit_time_200,Dx_ICD9_list,Dx_ICD10_list,Dx_name_clean,outcome_sepsis,outcome_copd_exac,outcome_acs_mi,outcome_stroke,outcome_ards,outcome_aki,outcome_pe,outcome_pneumonia_bacterial,outcome_pneumonia_viral,outcome_pneumonia_all,outcome_asthma_acute_exac,outcome_ahf,outcome_hospitalization,outcome_critical,outcome_discharge
0,98880961,98874959,1,1,82,F,White,Non-Hispanic/Non-Latino,EMS,36.2,81.0,18.0,96.0,122.0,86.0,3-Urgent,FALL,Inpatient,NaN,NaN,6.13,1.0,Skilled Nursing Facility,Medicare,Medicine,873.20,S01.21XA,"Laceration of nose, initial encounter",2283-03-02T07:36:59Z,2283-03-02T07:36:59Z,2283-03-02T12:29:59Z,2283-03-02T12:29:59Z,2283-03-02T13:44:59Z,2083-03-02 07:36:59,2083-03-02 07:36:59,2083-03-02 13:44:59,2083-03-02 12:29:59,2083-03-02 12:29:59,[87320],[S0121XA],"laceration of nose, initial encounter",False,False,False,False,False,False,False,False,False,False,False,False,1,0,0
1,99121566,99354408,1,2,59,F,Other,Non-Hispanic/Non-Latino,EMS,36.8,98.0,14.0,98.0,103.0,66.0,2-Emergent,"ALTERED MENTAL STATUS,SHORTNESS OF BREATH",Inpatient,4841.0,ICU,9.53,7.0,Skilled Nursing Facility,NaN,Emergency Medicine,780.97,R41.82,"Altered mental status, unspecified altered mental status type",2239-08-22T01:13:51Z,2239-08-22T01:16:51Z,2239-08-22T07:20:51Z,2239-08-22T07:20:51Z,2239-08-22T10:45:51Z,2039-08-22 01:13:51,2039-08-22 01:16:51,2039-08-22 10:45:51,2039-08-22 07:20:51,2039-08-22 07:20:51,[78097],[R4182],"altered mental status, unspecified altered mental status type",False,False,False,False,False,False,False,False,False,False,False,False,1,0,0
2,99121566,99121037,2,2,63,F,Other,Non-Hispanic/Non-Latino,EMS,37.2,96.0,18.0,98.0,91.0,61.0,3-Urgent,"ABNORMAL LAB,GENERAL WEAKNESS",ICU,NaN,NaN,10.52,16.0,Skilled Nursing Facility,NaN,Critical Care,"038.9, 785.52, 995.92","A41.9, R65.21",Septic shock (CMS-HCC),2240-03-11T03:06:51Z,2240-03-11T03:43:51Z,2240-03-11T08:31:51Z,2240-03-11T08:31:51Z,2240-03-11T13:37:51Z,2040-03-11 03:06:51,2040-03-11 03:43:51,2040-03-11 13:37:51,2040-03-11 08:31:51,2040-03-11 08:31:51,"[0389, 78552, 99592]","[A419, R6521]",septic shock,True,False,False,False,False,False,False,False,False,False,False,False,0,1,0
3,99608739,99327434,1,2,90,F,White,Non-Hispanic/Non-Latino,Self,NaN,69.0,16.0,98.0,107.0,42.0,3-Urgent,POSSIBLE DVT,Discharge,6704.0,Discharge,4.38,NaN,Home/Work (includes foster care),Medicare,NaN,729.81,M79.89,Leg swelling,2296-08-31T16:43:05Z,2296-08-31T17:56:05Z,2296-08-31T20:56:05Z,NaN,2296-08-31T21:06:05Z,2096-08-31 16:43:05,2096-08-31 17:56:05,2096-08-31 21:06:05,2096-08-31 20:56:05,NaT,[72981],[M7989],leg swelling,False,False,False,False,False,False,False,False,False,False,False,False,0,0,1
4,99608739,99502864,2,2,89,F,White,Non-Hispanic/Non-Latino,Self,36.4,65.0,17.0,98.0,103.0,57.0,3-Urgent,URINARY RETENTION,Discharge,NaN,Discharge,4.45,NaN,Home/Work (includes foster care),Medicare,NaN,788.20,R33.9,Urinary retention,2297-06-07T05:08:05Z,2297-06-07T05:59:05Z,2297-06-07T08:55:05Z,NaN,2297-06-07T09:35:05Z,2097-06-07 05:08:05,2097-06-07 05:59:05,2097-06-07 09:35:05,2097-06-07 08:55:05,NaT,[78820],[R339],urinary retention,False,False,False,False,False,False,False,False,False,False,False,False,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,3,5,41,F,Other,Non-Hispanic/Non-Latino,Self,36.8,86.0,18.0,100.0,134.0,76.0,2-Emergent,"MULTIPLE COMPLAINTS,SUICIDAL",Discharge,67.0,NaN,4.58,NaN,Home/Wor

In [268]:
outcome_cols = ["outcome_hospitalization", "outcome_critical", "outcome_discharge"]

proportions = visits_history_copy[outcome_cols].mean()

print(proportions)

outcome_hospitalization    0.247920
outcome_critical           0.023364
outcome_discharge          0.593369
dtype: float64


In [269]:

visits_history_copy["Hours_to_next_visit"] = pd.to_numeric(visits_history_copy["Hours_to_next_visit"])

visits_history_copy["outcome_ed_revisit_72h"] = ((visits_history_copy["Hours_to_next_visit"] < 72)).astype(int)
visits_history_copy["outcome_ed_revisit_168h"] = ((visits_history_copy["Hours_to_next_visit"] < 168)).astype(int)
visits_history_copy["outcome_ed_revisit_720h"] = ((visits_history_copy["Hours_to_next_visit"] < 720)).astype(int)

proportions = visits_history_copy[[
    "outcome_ed_revisit_72h",
    "outcome_ed_revisit_168h",
    "outcome_ed_revisit_720h"
]].mean()

print(proportions)

outcome_ed_revisit_72h     0.038299
outcome_ed_revisit_168h    0.073844
outcome_ed_revisit_720h    0.178908
dtype: float64


In [270]:
def check_visit_outcome(visit_row, pmh_indexed, codes, keywords, discharge_col):
    discharge_time = visit_row[discharge_col]
    mrn = str(visit_row['MRN'])
    
    if pd.isna(discharge_time):
        return 0
    
    if mrn not in pmh_indexed.index:
        return 0

    try:
        patient_pmh = pmh_indexed.loc[[mrn]]
    except KeyError:
        return 0
    
    if patient_pmh.empty:
        return 0
    
    time_diff_hours = (patient_pmh['Noted_date_200'] - discharge_time).dt.total_seconds() / 3600
    
    records_in_window = patient_pmh[(time_diff_hours >= 0) & (time_diff_hours <= 72)]
    
    if records_in_window.empty:
        return 0
    
    code_match = False
    if codes:
        if 'ICD_9_Code' in records_in_window.columns:
            icd9_matches = records_in_window['ICD_9_Code'].astype(str).str.upper().str.startswith(tuple(codes), na=False)
            if icd9_matches.any():
                code_match = True
        
        if 'ICD_10_Code' in records_in_window.columns and not code_match:
            icd10_matches = records_in_window['ICD_10_Code'].astype(str).str.upper().str.startswith(tuple(codes), na=False)
            if icd10_matches.any():
                code_match = True
    
    keyword_match = False
    if keywords and 'Desc10' in records_in_window.columns:
        keyword_pattern = '|'.join(keywords)
        keyword_matches = records_in_window['Desc10'].astype(str).str.lower().str.contains(
            keyword_pattern, regex=True, na=False
        )
        if keyword_matches.any():
            keyword_match = True
    
    return int(code_match or keyword_match)

In [ ]:
def preprocess_data(visits_df, pmh_df):
   
    visits_processed = visits_df.copy()
    pmh_processed = pmh_df.copy()
    
    discharge_time_candidates = ['Departure_time_200', 'discharge_time', 'outtime', 'end_time', 'Arrival_time']
    discharge_col = None
    for candidate in discharge_time_candidates:
        if candidate in visits_processed.columns:
            discharge_col = candidate
            break
    
    if discharge_col is None:
        raise KeyError(f"Discharge time column not found. Please check column names. Available columns: {list(visits_processed.columns)}")
    
    visits_processed[discharge_col] = pd.to_datetime(visits_processed[discharge_col])
    pmh_processed['Noted_date_200'] = pd.to_datetime(pmh_processed['Noted_date_200'])
    
    visits_processed['MRN'] = visits_processed['MRN'].astype(str)
    pmh_processed['MRN'] = pmh_processed['MRN'].astype(str)
    
    pmh_indexed = pmh_processed.set_index('MRN')
    
    diagnosis_columns = ['ICD_9_Code', 'ICD_10_Code', 'Desc10']
    for col in diagnosis_columns:
        if col in pmh_indexed.columns:
            pmh_indexed[col] = pmh_indexed[col].fillna('').astype(str)
    
    return visits_processed, pmh_indexed, discharge_col

In [272]:
def vectorized_outcome_detection(visits_df, pmh_indexed, disease_defs, discharge_col):
    
    results = {}
    total_visits = len(visits_df)

    available_patients = set(pmh_indexed.index)
    visits_patients = set(visits_df['MRN'].astype(str))
    overlap_patients = available_patients.intersection(visits_patients)
    
    for disease, disease_def in disease_defs.items():
    
        codes = [str(c).upper() for c in disease_def.get("codes", [])] if disease_def.get("codes") else []
        keywords = [str(k).lower() for k in disease_def.get("keywords", [])] if disease_def.get("keywords") else []
        
        disease_results = []
        positive_count = 0
        
        for idx, visit_row in visits_df.iterrows():
            result = check_visit_outcome(visit_row, pmh_indexed, codes, keywords, discharge_col)
            disease_results.append(result)
            if result == 1:
                positive_count += 1
        
        results[disease] = disease_results
    
    return results

In [273]:
def merge_based_approach(visits_df, pmh_indexed, disease_defs, discharge_col):
    pmh_reset = pmh_indexed.reset_index()
    
    visits_work = visits_df[['MRN', discharge_col]].copy().reset_index()
    visits_work['visit_index'] = visits_work.index
    visits_work['MRN'] = visits_work['MRN'].astype(str)
    
    merged = visits_work.merge(pmh_reset, on='MRN', how='left')
    
    merged['time_diff_hours'] = (merged['Noted_date_200'] - merged[discharge_col]).dt.total_seconds() / 3600
    
    valid_records = merged[
        (merged['time_diff_hours'] >= 0) & 
        (merged['time_diff_hours'] <= 72) &
        (merged['time_diff_hours'].notna())
    ].copy()
    
    results = {}
 
    for disease, disease_def in disease_defs.items():
        
        codes = [str(c).upper() for c in disease_def.get("codes", [])] if disease_def.get("codes") else []
        keywords = [str(k).lower() for k in disease_def.get("keywords", [])] if disease_def.get("keywords") else []
        
        if valid_records.empty:
            results[disease] = [0] * len(visits_df)
            continue
        
        code_match_mask = pd.Series(False, index=valid_records.index)
        if codes:
            if 'ICD_9_Code' in valid_records.columns:
                icd9_match = valid_records['ICD_9_Code'].astype(str).str.upper().str.startswith(tuple(codes), na=False)
                code_match_mask |= icd9_match
            
            if 'ICD_10_Code' in valid_records.columns:
                icd10_match = valid_records['ICD_10_Code'].astype(str).str.upper().str.startswith(tuple(codes), na=False)
                code_match_mask |= icd10_match
        
        keyword_match_mask = pd.Series(False, index=valid_records.index)
        if keywords and 'Desc10' in valid_records.columns:
            keyword_pattern = '|'.join(keywords)
            keyword_match_mask = valid_records['Desc10'].astype(str).str.lower().str.contains(
                keyword_pattern, regex=True, na=False
            )
        
        overall_match = code_match_mask | keyword_match_mask
        
        visit_matches = valid_records.loc[overall_match, 'visit_index'].unique() if overall_match.any() else []
        
        final_results = [0] * len(visits_df)
        for visit_idx in visit_matches:
            final_results[visit_idx] = 1
        
        results[disease] = final_results
    
    return results

In [274]:
def apply_outcome_detection(visits_df, pmh_df, disease_defs, method='vectorized'):
    
    visits_processed, pmh_indexed, discharge_col = preprocess_data(visits_df, pmh_df)
    
    if method == 'merge':
        results = merge_based_approach(visits_processed, pmh_indexed, disease_defs, discharge_col)
    else:
        results = vectorized_outcome_detection(visits_processed, pmh_indexed, disease_defs, discharge_col)
    
    visits_result = visits_df.copy()
    for disease, disease_results in results.items():
        column_name = f"outcome_{disease}_3d"
        visits_result[column_name] = disease_results
    
    return visits_result

In [275]:
outcome_cols_regular = [
    'outcome_sepsis',
    'outcome_copd_exac', 
    'outcome_acs_mi',
    'outcome_stroke',
    'outcome_ards',
    'outcome_aki',
    'outcome_pe',
    'outcome_pneumonia_bacterial',
    'outcome_pneumonia_viral',
    'outcome_pneumonia_all',
    'outcome_asthma_acute_exac',
    'outcome_ahf',
    'outcome_hospitalization',
    'outcome_critical',
    'outcome_discharge',
    'outcome_ed_revisit_72h',
    'outcome_ed_revisit_168h',
    'outcome_ed_revisit_720h'
]

existing_regular_cols = [col for col in outcome_cols_regular if col in visits_history_copy.columns]

merge_columns_regular = ['MRN', 'CSN'] + existing_regular_cols

df_patient_visits['subject_id'] = df_patient_visits['subject_id'].astype(str)
df_patient_visits['stay_id'] = df_patient_visits['stay_id'].astype(str)
visits_history_copy['MRN'] = visits_history_copy['MRN'].astype(str)
visits_history_copy['CSN'] = visits_history_copy['CSN'].astype(str)

df_patient_visits = df_patient_visits.merge(
    visits_history_copy[merge_columns_regular],
    left_on=['subject_id', 'stay_id'],
    right_on=['MRN', 'CSN'],
    how='left'
).drop(columns=['MRN', 'CSN'])

In [276]:
df_patient_visits

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness,n_ed_30d,n_ed_90d,n_ed_365d,n_hosp_30d,n_hosp_90d,n_hosp_365d,n_icu_30d,n_icu_90d,n_icu_365d,n_med,n_medrecon,cci_MI,cci_CHF,cci_PVD,cci_Stroke,cci_Dementia,cci_Pulmonary,cci_Rheumatic,cci_PUD,cci_Liver1,cci_DM1,cci_DM2,cci_Paralysis,cci_Renal,cci_Cancer1,cci_Liver2,cci_Cancer2,cci_HIV,eci_HIV,eci_Alcohol,eci_BloodLoss,eci_Arrhythmia,eci_Pulmonary,eci_Coagulopathy,eci_CHF,eci_Anemia,eci_Depression,eci_DM2,eci_DM1,eci_Drugs,eci_FluidsLytes,eci_HTN2,eci_HTN1,eci_Hypothyroid,eci_Liver,eci_Lymphoma,eci_Tumor2,eci_Obesity,eci_NeuroOther,eci_Paralysis,eci_PUD,eci_PVD,eci_Psychoses,eci_PHTN,eci_Renal,eci_Rheumatic,eci_Tumor1,eci_Valvular,eci_WeightLoss,score_REMS,score_CART,score_NEWS,score_NEWS2,score_MEWS,score_CCI,ALBUMIN,ALKALINE PHOSPHATASE,ALT (SGPT),ANION GAP,AST (SGOT),BASOPHIL % (AUTO DIFF),"BILIRUBIN, TOTAL",BLOOD UREA NITROGEN (BUN),BLOOD: URINE (UA),"BNP, NT-PRO",Basophil Absolute (Combined),CHLORIDE,CO2,CREATININE,Calcium (Combined),EOSINOPHIL % (AUTO DIFF),Eosinophil Absolute (Combined),GLOBULIN,GLUCOSE,GLUCOSE: URINE (UA),HCT & Hemoglobin (Combined),HEMATOCRIT (HCT),HIGH SENSITIVITY TROPONIN,IMMATURE GRANULOCYTE % (AUTODIFF) WAM,"IMMATURE GRANULOCYTE, ABSOLUTE (AUTO DIFF) WAM",INFLUENZA A,INFLUENZA B,INR,KETONE: URINE (UA),LEUKOCYTE ESTERASE: URINE (UA),LIPASE,LYMPHOCYTE % (AUTO DIFF),Lymphocyte Absolute (Combined),MAGNESIUM,MEAN CORPUSCULAR VOLUME (MCV),MONOCYTE % (AUTO DIFF),Monocyte Absolute (Combined),NEUTROPHIL % (AUTO DIFF),"NEUTROPHIL, ABSOLUTE (AUTO DIFF)",NITRITE: URINE (UA),NRBC (Combined),PART. THROMBOPLASTIN TIME,"PCO2, Venous (Combined)",PH: URINE (UA),PLATELET COUNT (PLT),"PO2, Venous (Combined)","POC: BHCG, ISTAT QUANTITATIVE","POC: HCG, ISTAT QUALITATIVE","POC:BASE EXCESS, ISTAT",POC:GLUCOSE BY METER,"POC:HCO3 (V), ISTAT","POC:LACTATE, ISTAT","POC:O2 SATURATION, ISTAT VENOUS","POC:TCO2 (V), ISTAT","PROTEIN, TOTAL",PROTEIN: URINE (UA),PROTHROMBIN TIME,Potassium (Combined),RED BLOOD CELLS (RBC),RED CELL DISTRIBUTION WIDTH (RDW),RSV,SARS-COV-2 RNA,SPECIFIC GRAVITY: URINE (UA),Sodium (Combined),TROPONIN I,WHITE BLOOD CELLS (WBC),eGFR (Combined),"pH, Venous (Combined)",outcome_sepsis,outcome_copd_exac,outcome_acs_mi,outcome_stroke,outcome_ards,outcome_aki,outcome_pe,outcome_pneumonia_bacterial,outcome_pneumonia_viral,outcome_pneumonia_all,outcome_asthma_acute_exac,outcome_ahf,outcome_hospitalization,outcome_critical,outcome_discharge,outcome_ed_revisit_72h,outcome_ed_revisit_168h,outcome_ed_revisit_720h
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.000000,FALL,False,False,False,False,False,False,False,False,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,9,0,0,1,5,3.4,139.0,28.0,9.0,38.0,8.0,2.0,17.0,34.0,NaN,7.0,108.0,23.0,84.000000,9.250000,3.8,32.0,4.7,109.000000,34.0,17.500000,21.133333,NaN,5.0,4.0,NaN,NaN,NaN,34.0,NaN,NaN,13.1,1.11,NaN,70.033333,8.6,73.00,73.2,6.23,2.0,0.0,NaN,NaN,6.0,410.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.1,34.0,NaN,4.500000,3.016667,22.2,NaN,NaN,1.0200,140.0,17.0,8.1,NaN,NaN,False,False,False,False,False,False,False,False,False,False,False,False,1,0,0,0,0,0
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51

In [278]:
visits_result = apply_outcome_detection(visits_history_copy, pmh_history_copy, DISEASE_DEFS, method='vectorized')

outcome_cols_3d = [col for col in visits_result.columns if col.startswith("outcome_") and col.endswith("_3d")]

outcome_proportions = visits_result[outcome_cols_3d].mean()

print("3d Outcome proportions:")
print(outcome_proportions)

conflicting_3d_cols = [col for col in outcome_cols_3d if col in df_patient_visits.columns]
if conflicting_3d_cols:
    for col in conflicting_3d_cols:
        print(f"  - {col}")
    df_patient_visits = df_patient_visits.drop(columns=conflicting_3d_cols)

merge_columns_3d = ['MRN', 'CSN'] + outcome_cols_3d

visits_result['MRN'] = visits_result['MRN'].astype(str)
visits_result['CSN'] = visits_result['CSN'].astype(str)

print("Performing second merge: 3d outcome columns...")
df_patient_visits = df_patient_visits.merge(
    visits_result[merge_columns_3d],
    left_on=['subject_id', 'stay_id'],
    right_on=['MRN', 'CSN'],
    how='left'
).drop(columns=['MRN', 'CSN'])

display(df_patient_visits)

3d Outcome proportions:
outcome_sepsis_3d                 0.001647
outcome_copd_exac_3d              0.000608
outcome_acs_mi_3d                 0.001428
outcome_stroke_3d                 0.002095
outcome_ards_3d                   0.000008
outcome_aki_3d                    0.002999
outcome_pe_3d                     0.001242
outcome_pneumonia_bacterial_3d    0.000025
outcome_pneumonia_viral_3d        0.000498
outcome_pneumonia_all_3d          0.000524
outcome_asthma_acute_exac_3d      0.000431
outcome_ahf_3d                    0.001943
dtype: float64
Performing second merge: 3d outcome columns...


,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness,n_ed_30d,n_ed_90d,n_ed_365d,n_hosp_30d,n_hosp_90d,n_hosp_365d,n_icu_30d,n_icu_90d,n_icu_365d,n_med,n_medrecon,cci_MI,cci_CHF,cci_PVD,cci_Stroke,cci_Dementia,cci_Pulmonary,cci_Rheumatic,cci_PUD,cci_Liver1,cci_DM1,cci_DM2,cci_Paralysis,cci_Renal,cci_Cancer1,cci_Liver2,cci_Cancer2,cci_HIV,eci_HIV,eci_Alcohol,eci_BloodLoss,eci_Arrhythmia,eci_Pulmonary,eci_Coagulopathy,eci_CHF,eci_Anemia,eci_Depression,eci_DM2,eci_DM1,eci_Drugs,eci_FluidsLytes,eci_HTN2,eci_HTN1,eci_Hypothyroid,eci_Liver,eci_Lymphoma,eci_Tumor2,eci_Obesity,eci_NeuroOther,eci_Paralysis,eci_PUD,eci_PVD,eci_Psychoses,eci_PHTN,eci_Renal,eci_Rheumatic,eci_Tumor1,eci_Valvular,eci_WeightLoss,score_REMS,score_CART,score_NEWS,score_NEWS2,score_MEWS,score_CCI,ALBUMIN,ALKALINE PHOSPHATASE,ALT (SGPT),ANION GAP,AST (SGOT),BASOPHIL % (AUTO DIFF),"BILIRUBIN, TOTAL",BLOOD UREA NITROGEN (BUN),BLOOD: URINE (UA),"BNP, NT-PRO",Basophil Absolute (Combined),CHLORIDE,CO2,CREATININE,Calcium (Combined),EOSINOPHIL % (AUTO DIFF),Eosinophil Absolute (Combined),GLOBULIN,GLUCOSE,GLUCOSE: URINE (UA),HCT & Hemoglobin (Combined),HEMATOCRIT (HCT),HIGH SENSITIVITY TROPONIN,IMMATURE GRANULOCYTE % (AUTODIFF) WAM,"IMMATURE GRANULOCYTE, ABSOLUTE (AUTO DIFF) WAM",INFLUENZA A,INFLUENZA B,INR,KETONE: URINE (UA),LEUKOCYTE ESTERASE: URINE (UA),LIPASE,LYMPHOCYTE % (AUTO DIFF),Lymphocyte Absolute (Combined),MAGNESIUM,MEAN CORPUSCULAR VOLUME (MCV),MONOCYTE % (AUTO DIFF),Monocyte Absolute (Combined),NEUTROPHIL % (AUTO DIFF),"NEUTROPHIL, ABSOLUTE (AUTO DIFF)",NITRITE: URINE (UA),NRBC (Combined),PART. THROMBOPLASTIN TIME,"PCO2, Venous (Combined)",PH: URINE (UA),PLATELET COUNT (PLT),"PO2, Venous (Combined)","POC: BHCG, ISTAT QUANTITATIVE","POC: HCG, ISTAT QUALITATIVE","POC:BASE EXCESS, ISTAT",POC:GLUCOSE BY METER,"POC:HCO3 (V), ISTAT","POC:LACTATE, ISTAT","POC:O2 SATURATION, ISTAT VENOUS","POC:TCO2 (V), ISTAT","PROTEIN, TOTAL",PROTEIN: URINE (UA),PROTHROMBIN TIME,Potassium (Combined),RED BLOOD CELLS (RBC),RED CELL DISTRIBUTION WIDTH (RDW),RSV,SARS-COV-2 RNA,SPECIFIC GRAVITY: URINE (UA),Sodium (Combined),TROPONIN I,WHITE BLOOD CELLS (WBC),eGFR (Combined),"pH, Venous (Combined)",outcome_sepsis,outcome_copd_exac,outcome_acs_mi,outcome_stroke,outcome_ards,outcome_aki,outcome_pe,outcome_pneumonia_bacterial,outcome_pneumonia_viral,outcome_pneumonia_all,outcome_asthma_acute_exac,outcome_ahf,outcome_hospitalization,outcome_critical,outcome_discharge,outcome_ed_revisit_72h,outcome_ed_revisit_168h,outcome_ed_revisit_720h,outcome_sepsis_3d,outcome_copd_exac_3d,outcome_acs_mi_3d,outcome_stroke_3d,outcome_ards_3d,outcome_aki_3d,outcome_pe_3d,outcome_pneumonia_bacterial_3d,outcome_pneumonia_viral_3d,outcome_pneumonia_all_3d,outcome_asthma_acute_exac_3d,outcome_ahf_3d
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.000000,FALL,False,False,False,False,False,False,False,False,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,9,0,0,1,5,3.4,139.0,28.0,9.0,38.0,8.0,2.0,17.0,34.0,NaN,7.0,108.0,23.0,84.000000,9.250000,3.8,32.0,4.7,109.000000,34.0,17.500000,21.133333,NaN,5.0,4.0,NaN,NaN,NaN,34.0,NaN,NaN,13.1,1.11,NaN,70.033333,8.6,73.00,73.2,6.23,2.0,0.0,NaN,NaN,6.0,410.666667,NaN,NaN,NaN,NaN,NaN,N

In [ ]:
df_patient_visits['outcome_icu_transfer_12h'] = (df_patient_visits['time_to_icu_transfer_hours'] <= 12).astype(int)

print(df_patient_visits['outcome_icu_transfer_12h'].value_counts())

outcome_icu_transfer_12h
0    115633
1      2752
Name: count, dtype: int64


In [285]:
outcome_columns = [col for col in df_patient_visits.columns if col.startswith('outcome')]
for col in outcome_columns:
    print(f"  - {col}")

  - outcome_sepsis
  - outcome_copd_exac
  - outcome_acs_mi
  - outcome_stroke
  - outcome_ards
  - outcome_aki
  - outcome_pe
  - outcome_pneumonia_bacterial
  - outcome_pneumonia_viral
  - outcome_pneumonia_all
  - outcome_asthma_acute_exac
  - outcome_ahf
  - outcome_hospitalization
  - outcome_critical
  - outcome_discharge
  - outcome_ed_revisit_72h
  - outcome_ed_revisit_168h
  - outcome_ed_revisit_720h
  - outcome_sepsis_3d
  - outcome_copd_exac_3d
  - outcome_acs_mi_3d
  - outcome_stroke_3d
  - outcome_ards_3d
  - outcome_aki_3d
  - outcome_pe_3d
  - outcome_pneumonia_bacterial_3d
  - outcome_pneumonia_viral_3d
  - outcome_pneumonia_all_3d
  - outcome_asthma_acute_exac_3d
  - outcome_ahf_3d
  - outcome_icu_transfer_12h


In [287]:
outcome_types = [
    'sepsis', 'copd_exac', 'acs_mi', 'stroke', 'ards', 'aki', 'pe',
    'pneumonia_bacterial', 'pneumonia_viral', 'pneumonia_all',
    'asthma_acute_exac', 'ahf', 'copd_asthma'
]

for outcome_type in outcome_types:
    ed_outcome = f'outcome_{outcome_type}'
    three_day_outcome = f'outcome_{outcome_type}_3d'
    new_outcome = f'outcome_{outcome_type}_3d_new'
    
    if ed_outcome in df_patient_visits.columns and three_day_outcome in df_patient_visits.columns:
        df_patient_visits[new_outcome] = (df_patient_visits[ed_outcome] + df_patient_visits[three_day_outcome]).astype(int)

In [288]:
df_patient_visits

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP,CC,chiefcom_chest_pain,chiefcom_abdominal_pain,chiefcom_headache,chiefcom_shortness_of_breath,chiefcom_back_pain,chiefcom_cough,chiefcom_nausea_vomiting,chiefcom_fever_chills,chiefcom_syncope,chiefcom_dizziness,n_ed_30d,n_ed_90d,n_ed_365d,n_hosp_30d,n_hosp_90d,n_hosp_365d,n_icu_30d,n_icu_90d,n_icu_365d,n_med,n_medrecon,cci_MI,cci_CHF,cci_PVD,cci_Stroke,cci_Dementia,cci_Pulmonary,cci_Rheumatic,cci_PUD,cci_Liver1,cci_DM1,cci_DM2,cci_Paralysis,cci_Renal,cci_Cancer1,cci_Liver2,cci_Cancer2,cci_HIV,eci_HIV,eci_Alcohol,eci_BloodLoss,eci_Arrhythmia,eci_Pulmonary,eci_Coagulopathy,eci_CHF,eci_Anemia,eci_Depression,eci_DM2,eci_DM1,eci_Drugs,eci_FluidsLytes,eci_HTN2,eci_HTN1,eci_Hypothyroid,eci_Liver,eci_Lymphoma,eci_Tumor2,eci_Obesity,eci_NeuroOther,eci_Paralysis,eci_PUD,eci_PVD,eci_Psychoses,eci_PHTN,eci_Renal,eci_Rheumatic,eci_Tumor1,eci_Valvular,eci_WeightLoss,score_REMS,score_CART,score_NEWS,score_NEWS2,score_MEWS,score_CCI,ALBUMIN,ALKALINE PHOSPHATASE,ALT (SGPT),ANION GAP,AST (SGOT),BASOPHIL % (AUTO DIFF),"BILIRUBIN, TOTAL",BLOOD UREA NITROGEN (BUN),BLOOD: URINE (UA),"BNP, NT-PRO",Basophil Absolute (Combined),CHLORIDE,CO2,CREATININE,Calcium (Combined),EOSINOPHIL % (AUTO DIFF),Eosinophil Absolute (Combined),GLOBULIN,GLUCOSE,GLUCOSE: URINE (UA),HCT & Hemoglobin (Combined),HEMATOCRIT (HCT),HIGH SENSITIVITY TROPONIN,IMMATURE GRANULOCYTE % (AUTODIFF) WAM,"IMMATURE GRANULOCYTE, ABSOLUTE (AUTO DIFF) WAM",INFLUENZA A,INFLUENZA B,INR,KETONE: URINE (UA),LEUKOCYTE ESTERASE: URINE (UA),LIPASE,LYMPHOCYTE % (AUTO DIFF),Lymphocyte Absolute (Combined),MAGNESIUM,MEAN CORPUSCULAR VOLUME (MCV),MONOCYTE % (AUTO DIFF),Monocyte Absolute (Combined),NEUTROPHIL % (AUTO DIFF),"NEUTROPHIL, ABSOLUTE (AUTO DIFF)",NITRITE: URINE (UA),NRBC (Combined),PART. THROMBOPLASTIN TIME,"PCO2, Venous (Combined)",PH: URINE (UA),PLATELET COUNT (PLT),"PO2, Venous (Combined)","POC: BHCG, ISTAT QUANTITATIVE","POC: HCG, ISTAT QUALITATIVE","POC:BASE EXCESS, ISTAT",POC:GLUCOSE BY METER,"POC:HCO3 (V), ISTAT","POC:LACTATE, ISTAT","POC:O2 SATURATION, ISTAT VENOUS","POC:TCO2 (V), ISTAT","PROTEIN, TOTAL",PROTEIN: URINE (UA),PROTHROMBIN TIME,Potassium (Combined),RED BLOOD CELLS (RBC),RED CELL DISTRIBUTION WIDTH (RDW),RSV,SARS-COV-2 RNA,SPECIFIC GRAVITY: URINE (UA),Sodium (Combined),TROPONIN I,WHITE BLOOD CELLS (WBC),eGFR (Combined),"pH, Venous (Combined)",outcome_sepsis,outcome_copd_exac,outcome_acs_mi,outcome_stroke,outcome_ards,outcome_aki,outcome_pe,outcome_pneumonia_bacterial,outcome_pneumonia_viral,outcome_pneumonia_all,outcome_asthma_acute_exac,outcome_ahf,outcome_hospitalization,outcome_critical,outcome_discharge,outcome_ed_revisit_72h,outcome_ed_revisit_168h,outcome_ed_revisit_720h,outcome_sepsis_3d,outcome_copd_exac_3d,outcome_acs_mi_3d,outcome_stroke_3d,outcome_ards_3d,outcome_aki_3d,outcome_pe_3d,outcome_pneumonia_bacterial_3d,outcome_pneumonia_viral_3d,outcome_pneumonia_all_3d,outcome_asthma_acute_exac_3d,outcome_ahf_3d,outcome_icu_transfer_12h,outcome_sepsis_3d_new,outcome_copd_exac_3d_new,outcome_acs_mi_3d_new,outcome_stroke_3d_new,outcome_ards_3d_new,outcome_aki_3d_new,outcome_pe_3d_new,outcome_pneumonia_bacterial_3d_new,outcome_pneumonia_viral_3d_new,outcome_pneumonia_all_3d_new,outcome_asthma_acute_exac_3d_new,outcome_ahf_3d_new
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.000000,FALL,False,False,False,False,False,False,False,False,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,

In [289]:
df_patient_visits.to_csv('/Users/xinniemai/Desktop/mcmed_data/master_dataset.csv', index=False)